# CLI Workflows & Automation

qb-compiler ships with a command-line interface (`qbc`) for common compilation and
analysis tasks. This notebook demonstrates CLI usage via `subprocess` calls and shows
how to integrate `qbc` into CI/CD pipelines and automated workflows.

**Note:** This notebook runs CLI commands via `subprocess`. The `qbc` CLI must be
installed (`pip install qb-compiler[cli]` or from source).

Commands covered:
- `qbc doctor` — environment check
- `qbc info` — version and backend listing
- `qbc preflight` — quick viability check
- `qbc analyze` — detailed circuit analysis
- `qbc compile` — compilation with `--receipt`
- `qbc diff` — backend comparison
- `qbc calibration show` — calibration data inspection
- Automation patterns: batch preflight, CI/CD hooks

In [ ]:
import subprocess
import json
import tempfile
from pathlib import Path


def run_qbc(*args, check=True):
    """Run a qbc CLI command and return stdout."""
    result = subprocess.run(
        ["qbc", *args],
        capture_output=True,
        text=True,
        timeout=60,
    )
    if check and result.returncode != 0:
        print(f"STDERR: {result.stderr}")
    print(result.stdout)
    return result

## 1. qbc doctor and qbc info

`qbc doctor` verifies your quantum development environment: Python version, Qiskit,
IBM credentials, calibration snapshots, and optional dependencies.
`qbc info` shows the installed version and lists all configured backends.

In [ ]:
run_qbc("doctor")
print("\n" + "=" * 60 + "\n")
run_qbc("info");

## 2. Create test circuits

The CLI commands operate on QASM files. Let's create a GHZ circuit to use throughout.

In [ ]:
from qiskit import QuantumCircuit

work_dir = Path(tempfile.mkdtemp(prefix="qbc_demo_"))

qc = QuantumCircuit(4, name="ghz_4")
qc.h(0)
for i in range(3):
    qc.cx(i, i + 1)
qc.measure_all()

qasm_path = work_dir / "ghz_4.qasm"
qasm_path.write_text(qc.qasm())

print(f"Circuit saved to: {qasm_path}")
print(f"\nQASM content:")
print(qasm_path.read_text())

## 3. qbc preflight — quick viability check

Returns one of three verdicts: **VIABLE**, **CAUTION**, or **DO NOT RUN**.

```
qbc preflight <circuit.qasm> --backend <backend> [--seeds N]
```

In [ ]:
run_qbc("preflight", str(qasm_path), "--backend", "ibm_fez");

## 4. qbc analyze and qbc compile

`qbc analyze` provides circuit type detection, gate breakdown, signal-to-noise ratio, and
actionable suggestions. `qbc compile` compiles with optional `--receipt` for metadata.

Strategies: `fidelity_optimal` (default), `depth_optimal`, `budget_optimal`

In [ ]:
run_qbc("analyze", str(qasm_path), "--backend", "ibm_fez");

In [ ]:
run_qbc(
    "compile", str(qasm_path),
    "--backend", "ibm_fez",
    "--strategy", "fidelity_optimal",
    "--receipt",
)

# Check if a receipt was generated
receipt_path = qasm_path.with_suffix(".receipt.json")
if receipt_path.exists():
    receipt = json.loads(receipt_path.read_text())
    print("Compilation receipt:")
    print(json.dumps(receipt, indent=2))
else:
    print("No receipt generated (compile may have been skipped).")

## 5. qbc diff and qbc calibration show

`qbc diff` compares circuit performance on two backends side by side.
`qbc calibration show` displays the calibration summary for a backend.

In [ ]:
run_qbc(
    "diff", str(qasm_path),
    "--backend", "ibm_fez",
    "--vs", "ibm_torino",
);

In [ ]:
for backend in ["ibm_fez", "ibm_torino", "ibm_marrakesh"]:
    print(f"=== {backend} ===")
    run_qbc("calibration", "show", backend)
    print()

## 6. Scripting: batch preflight checks

Run viability checks across multiple circuits and backends to find the best
circuit-backend pairings before spending QPU time.

In [ ]:
# Create additional test circuits
for name, builder in [
    ("bell", lambda: (q := QuantumCircuit(2, name="bell"), q.h(0), q.cx(0, 1), q.measure_all(), q)[-1]),
    ("ghz_8", lambda: (q := QuantumCircuit(8, name="ghz_8"), q.h(0), [q.cx(i, i+1) for i in range(7)], q.measure_all(), q)[-1]),
]:
    circ = builder()
    (work_dir / f"{name}.qasm").write_text(circ.qasm())

# Batch preflight
backends = ["ibm_fez", "ibm_torino"]
circuit_names = ["bell", "ghz_4", "ghz_8"]

print(f"{'Circuit':>15s} | {'Backend':>12s} | Result")
print("-" * 50)

for name in circuit_names:
    qasm_file = work_dir / f"{name}.qasm"
    for backend in backends:
        result = subprocess.run(
            ["qbc", "preflight", str(qasm_file), "--backend", backend],
            capture_output=True, text=True, timeout=60,
        )
        status = "ERROR"
        for line in result.stdout.splitlines():
            if "Status:" in line:
                status = line.split("Status:")[1].strip()
                break
        print(f"{name:>15s} | {backend:>12s} | {status}")

## 7. CI/CD integration patterns

### Pre-commit hook

```bash
#!/bin/bash
# .git/hooks/pre-commit — block commits with non-viable circuits

QASM_FILES=$(git diff --cached --name-only --diff-filter=ACM | grep '\.qasm$')
[ -z "$QASM_FILES" ] && exit 0

echo "Running qbc preflight on staged QASM files..."
FAILED=0

for f in $QASM_FILES; do
    OUTPUT=$(qbc preflight "$f" --backend ibm_fez 2>&1)
    if echo "$OUTPUT" | grep -q "DO NOT RUN"; then
        echo "FAIL: $f is not viable on ibm_fez"
        FAILED=1
    fi
done

[ $FAILED -ne 0 ] && echo "Commit blocked: non-viable circuits." && exit 1
```

### GitHub Actions workflow

```yaml
# .github/workflows/quantum-ci.yml
name: Quantum Circuit CI
on:
  push:
    paths: ['**.qasm', '**.py']
jobs:
  preflight:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.11'
      - run: pip install qb-compiler[cli]
      - run: qbc doctor
      - name: Preflight all QASM files
        run: |
          find . -name '*.qasm' | while read f; do
            echo "Checking: $f"
            qbc preflight "$f" --backend ibm_fez || true
          done
      - name: Compile with receipts
        run: |
          find . -name '*.qasm' | while read f; do
            qbc compile "$f" --backend ibm_fez --receipt || true
          done
      - uses: actions/upload-artifact@v4
        with:
          name: compilation-receipts
          path: '**/*.receipt.json'
```

## 8. Automated compilation pipeline

Combine multiple `qbc` commands into a pipeline that processes a directory of QASM
files and generates a summary report.

In [ ]:
def compilation_pipeline(circuit_dir, backend="ibm_fez"):
    """Run a full compilation pipeline on all QASM files in a directory."""
    circuit_dir = Path(circuit_dir)
    qasm_files = sorted(circuit_dir.glob("*.qasm"))

    if not qasm_files:
        print(f"No QASM files found in {circuit_dir}")
        return []

    results = []
    for qasm_file in qasm_files:
        entry = {"file": qasm_file.name, "backend": backend}

        result = subprocess.run(
            ["qbc", "preflight", str(qasm_file), "--backend", backend],
            capture_output=True, text=True, timeout=60,
        )
        for line in result.stdout.splitlines():
            if "Status:" in line:
                entry["status"] = line.split("Status:")[1].strip()
            if "Estimated fidelity:" in line:
                entry["fidelity"] = line.split(":")[1].strip()

        if entry.get("status") in ("VIABLE", "CAUTION"):
            compile_result = subprocess.run(
                ["qbc", "compile", str(qasm_file), "--backend", backend, "--receipt"],
                capture_output=True, text=True, timeout=60,
            )
            entry["compiled"] = compile_result.returncode == 0
        else:
            entry["compiled"] = False

        results.append(entry)
    return results


pipeline_results = compilation_pipeline(work_dir, backend="ibm_fez")
print("Pipeline results:")
print(json.dumps(pipeline_results, indent=2))

# Clean up
import shutil
shutil.rmtree(work_dir, ignore_errors=True)
print(f"\nCleaned up {work_dir}")

## CLI command reference

| Command | Purpose | Key flags |
|---------|---------|----------|
| `qbc doctor` | Environment check | — |
| `qbc info` | Version + backends | — |
| `qbc preflight` | Quick viability check | `--backend`, `--seeds` |
| `qbc analyze` | Detailed analysis | `--backend`, `--seeds` |
| `qbc compile` | Compile circuit | `--backend`, `--strategy`, `--receipt`, `--output` |
| `qbc diff` | Compare two backends | `--backend`, `--vs`, `--seeds` |
| `qbc calibration show` | Calibration summary | `<backend>` |